In [ ]:
# Loading Libraries
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.filterwarnings("ignore")
#from dfply import *
#import pyreadstat
import scorecardpy as sc
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
import scipy as sp
import math
import re
import requests
import random
import itertools
from numpy import mean, std
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
import statsmodels.api as sm
from sklearn import tree
from sklearn.datasets import make_classification
import matplotlib.ticker as ticker
from matplotlib.ticker import NullFormatter
from sklearn import preprocessing
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn import svm
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier
from sklearn.svm import SVC
from sklearn.ensemble import BaggingClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn.metrics import f1_score
from sklearn.metrics import jaccard_score
from sklearn.metrics import log_loss
from sklearn.metrics import roc_auc_score
from sklearn.metrics import classification_report, confusion_matrix
#from sklearn.metrics import ConfusionMatrixDisplay,plot_confusion_matrix
from sklearn.model_selection import GridSearchCV
import itertools
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas.core.algorithms as algos
from pandas import Series
import scipy.stats.stats as stats
import traceback
import string
pd.set_option('display.float_format', lambda x: '%.3f' % x)

In [ ]:
from sklearn.model_selection import train_test_split
# Split base into train and test set based on 80% - 20% split

df_train = base.sample(frac = 0.80)
df_test = base.drop(df_train.index)

print('Size of Training Set : {} and Test Set : {}'.format(len(df_train),len(df_test)))

In [ ]:
### Missing Value Imputation and Data Manipulation
## First the percentage of missing values in the data and if it is more than 20 or 30%, drop that column
### Imputation depends upon the data, it can be using Business logic, median values for continuous variables or mode values for categorical variables

df_train['Age'] = df_train['Age'].fillna(df_train['Age'].median())
df_train['Category'] = df_train['Category'].fillna(df_train['Category'].mode())

### Outlier Treatment

In [ ]:
outliers = {}
for x in cols:
    if df_train[x].dtype =='object':
        pass
    else:
        median_value = df_train[x].median()
        ul,ll = np.percentile(df_train.loc[:,x],[95,5])
        q75,q25 = np.percentile(df_train.loc[:,x],[75,25])
        intr_qr = q75-q25

        maximum = q75+(1.5*intr_qr)
        minimum = q25-(1.5*intr_qr)

        outliers.update({x : [minimum, maximum, median_value]})

        df_train.loc[df_train[x] > maximum,x] = ul
        df_test.loc[df_test[x] > maximum,x] = ul

### VIF

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
max1=11
cols=list(num_cols)
while(max1>10):
    vif = pd.DataFrame()
    vif['Features'] = X_train[cols].columns
    vif['VIF'] = [variance_inflation_factor(X_train[cols].values, i) for i in range(X_train[cols].shape[1])]    
    vif['VIF'] = round(vif['VIF'], 2)
    vif = vif.sort_values(by = "VIF", ascending = False)
    max1=vif['VIF'].max()
    print(max1)
    print(vif)
    cols.remove(list(vif[vif['VIF']==vif['VIF'].max()].Features)[0])

### Feature Selection in Classification can be done using different methods
1. Information Value/Weight of Evidence
2. Information Gain 

In [ ]:
# IV using Scorecard
import scorecardpy as sc
%matplotlib inline
# woe binning ------
for i in df_train.columns[1:]:
    try:
        bins = sc.woebin(df_train[[i,"Target"]], y="Target")
        sc.woebin_plot(bins)
    except:
        print('Error:' + i)

In [ ]:
# Manually Calculating IV

def iv_woe(data, target, bins, show_woe=False):

    #Empty Dataframe
    newDF,woeDF = pd.DataFrame(), pd.DataFrame()

    #Extract Column Names
    cols = data.columns

    #Run WOE and IV on all the independent variables
    for ivars in cols[~cols.isin([target])]:
        if (data[ivars].dtype.kind in 'bifc') and (len(np.unique(data[ivars]))>10):
            binned_x = pd.qcut(data[ivars], bins,  duplicates='drop')
            d0 = pd.DataFrame({'x': binned_x, 'y': data[target]})
        else:
            d0 = pd.DataFrame({'x': data[ivars], 'y': data[target]})
        d = d0.groupby("x", as_index=False).agg({"y": ["count", "sum"]})
        d.columns = ['Cutoff', 'N', 'Events']
        d['% of Events'] = np.maximum(d['Events'], 0.5) / d['Events'].sum()
        d['Non-Events'] = d['N'] - d['Events']
        d['% of Non-Events'] = np.maximum(d['Non-Events'], 0.5) / d['Non-Events'].sum()
        d['WoE'] = np.log(d['% of Events']/d['% of Non-Events'])
        d['IV'] = d['WoE'] * (d['% of Events'] - d['% of Non-Events'])
        d.insert(loc=0, column='Variable', value=ivars)
        print("Information value of " + ivars + " is " + str(round(d['IV'].sum(),6)))
        temp =pd.DataFrame({"Variable" : [ivars], "IV" : [d['IV'].sum()]}, columns = ["Variable", "IV"])
        newDF=pd.concat([newDF,temp], axis=0)
        woeDF=pd.concat([woeDF,d], axis=0)

        #Show WOE Table
        if show_woe == True:
            print(d)
    return newDF, woeDF

iv, woe = iv_woe(data = df_train, target = 'MF_Target', bins = 4, show_woe = True)

In [ ]:
def calc_entropy(column):
    """
    Calculate entropy given a series, list, or numpy array.
    """
    # Compute the counts of each given value in the column
    counts = np.bincount(column)
    # Divide by the total column length to get a probability
    probabilities = counts / len(column)

    # Initialize the entropy to 0
    entropy = 0
    # Loop through the probabilities, and add each one to the total entropy
    for prob in probabilities:
        if prob > 0:
            # use log from math and set base to 2
            entropy += prob * math.log(prob, 2)

    return -entropy

def calc_information_gain(data, split_name, target_name):
    try:
        """
        Calculate information gain given a data set, column to split on and target.
        """
        # Calculate the original entropy
        original_entropy = calc_entropy(data[target_name])


        # Find the unique values in the column
        values = data[split_name].unique()


        # Make two subsets of the data, based on the unique values
        #     print(data[data[split_name] == values[0]])
        #     print()
        #     print(data[data[split_name] == values[1]])


        print(split_name, ':', values)
        left_split = (data[data[split_name] == values[0]])
        right_split = (data[data[split_name] == values[1]])


        # Loop through the splits and calculate the subset entropies
        to_subtract = 0
        for subset in [left_split, right_split]:
            prob = (subset.shape[0] / data.shape[0])
            to_subtract += prob * calc_entropy(subset[target_name])


        # Return information gain
        return original_entropy - to_subtract
    except:
        return 0


target = 'Target'
dict_inf_gain = {i: format(calc_information_gain(df_train, i, target), 'f') for i in df_train.columns}
df_inf_gain = pd.DataFrame.from_dict(dict_inf_gain, orient = 'index', columns = ['Inf Gain'])

###  Encoding

In [ ]:
cat_vars = [i for i in df_train.columns[1:] if df_train[i].dtype == 'object']

for var in cat_vars:
    cat_list='var'+'_'+var
    cat_list = pd.get_dummies(df_train[var], prefix=var)
    #cat_list = cat_list.drop(cat_list.columns[0],axis = 1)
    df_train=df_train.join(cat_list)

df_train=df_train.drop(cat_vars,axis=1)


for var in cat_vars:
    cat_list='var'+'_'+var
    cat_list = pd.get_dummies(df_test[var], prefix=var)
    #cat_list = cat_list.drop(cat_list.columns[0],axis = 1)
    df_test=df_test.join(cat_list)

df_test=df_test.drop(cat_vars,axis=1)

### KS Table

In [ ]:
def ks(data=None,target=None, prob=None):
    data['target0'] = 1 - data[target]
    data['bucket'] = pd.qcut(data[prob], 10, duplicates='drop')
    grouped = data.groupby('bucket', as_index = False)
    kstable = pd.DataFrame()
    kstable['min_prob'] = grouped.min()[prob]
    kstable['max_prob'] = grouped.max()[prob]
    kstable['events'] = grouped.sum()[target]
    kstable['nonevents'] = grouped.sum()['target0']
    kstable = kstable.sort_values(by="min_prob", ascending=False).reset_index(drop = True)
    kstable['event_rate'] = (kstable.events / data[target].sum()).apply('{0:.2%}'.format)
    kstable['nonevent_rate'] = (kstable.nonevents / data['target0'].sum()).apply('{0:.2%}'.format)
    kstable['cum_eventrate']=(kstable.events / data[target].sum()).cumsum()
    kstable['cum_noneventrate']=(kstable.nonevents / data['target0'].sum()).cumsum()
    kstable['KS'] = np.round(kstable['cum_eventrate']-kstable['cum_noneventrate'], 3) * 100 #Formating
    kstable['cum_eventrate']= kstable['cum_eventrate'].apply('{0:.2%}'.format)
    kstable['cum_noneventrate']= kstable['cum_noneventrate'].apply('{0:.2%}'.format)
    kstable.index = range(1,11)
    kstable.index.rename('Decile', inplace=True)
    pd.set_option('display.max_columns', 9)
    #print(kstable)
    #Display KS
    from colorama import Fore
    print(Fore.RED + "KS is " + str(kstable['KS'].max())+"%"+ " at decile " + str((kstable.index[kstable['KS']==kstable['KS'].max()][0])))
    return(kstable)

### Model Building

In [ ]:
X_train_1 = df_train.drop(columns = ['Target'])
X_test_1 = df_test.drop(columns = ['Target'])

Y_train_1 = df_train['Target']
Y_test_1 = df_test['Target']

print(X_train_1.shape)
print(X_test_1.shape)
print(Y_train_1.shape)
print(Y_test_1.shape)

### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

log_regression = LogisticRegression(class_weight={1:5})

#fit the model using the training data
log_regression.fit(X_train_1,Y_train_1)

#use model to make predictions on test data
y_pred_tr_prob = log_regression.predict_proba(X_train_1)[:,1]
y_pred_ts_prob = log_regression.predict_proba(X_test_1)[:,1]

y_pred_tr = log_regression.predict(X_train_1)
y_pred_ts = log_regression.predict(X_test_1)

print('\n AUC Score for Train set\t:\t',roc_auc_score(Y_train_1,y_pred_tr_prob))
print('\n AUC Score for Test set\t:\t',roc_auc_score(Y_test_1,y_pred_ts_prob))

print('\n Accuracy_train\t:\t',accuracy_score(Y_train_1,y_pred_tr))
print('\n Accuracy_test\t:\t',accuracy_score(Y_test_1,y_pred_ts))

print('\n confusion_matrix_train\n',confusion_matrix(Y_train_1,y_pred_tr))
print('\n confusion_matrix_test\n',confusion_matrix(Y_test_1,y_pred_ts))

print('\n precision_score_train\t:\t',precision_score(Y_train_1,y_pred_tr))
print('\n precision_score_test\t:\t',precision_score(Y_test_1,y_pred_ts))

print('\n recall_score_train\t:\t',recall_score(Y_train_1,y_pred_tr))
print('\n recall_score_test\t:\t',recall_score(Y_test_1,y_pred_ts))

print(pd.DataFrame.from_dict(zip(list(X_train_1.columns),log_regression.coef_[0])))
print(log_regression.intercept_[0])


kstable_train=ks(pd.concat([pd.DataFrame(Y_train_1).reset_index(drop = True),pd.DataFrame(y_pred_tr_prob)], axis = 1),'Target',0)
kstable_test=ks(pd.concat([pd.DataFrame(Y_test_1).reset_index(drop = True),pd.DataFrame(y_pred_ts_prob)], axis = 1),'Target',0)

### StatsModel Logistic Regression

In [ ]:
import statsmodels.api as sm
logit_model=sm.Logit(y_train, X_train1)
result=logit_model.fit()
print(result.summary())

#use model to make predictions on test data
y_pred_tr_prob = result.predict_proba(X_train_1)[:,1]
y_pred_ts_prob = result.predict_proba(X_test_1)[:,1]

y_pred_tr = result.predict(X_train_1)
y_pred_ts = result.predict(X_test_1)

print('\n AUC Score for Train set\t:\t',roc_auc_score(Y_train_1,y_pred_tr_prob))
print('\n AUC Score for Test set\t:\t',roc_auc_score(Y_test_1,y_pred_ts_prob))

print('\n Accuracy_train\t:\t',accuracy_score(Y_train_1,y_pred_tr))
print('\n Accuracy_test\t:\t',accuracy_score(Y_test_1,y_pred_ts))

print('\n confusion_matrix_train\n',confusion_matrix(Y_train_1,y_pred_tr))
print('\n confusion_matrix_test\n',confusion_matrix(Y_test_1,y_pred_ts))

print('\n precision_score_train\t:\t',precision_score(Y_train_1,y_pred_tr))
print('\n precision_score_test\t:\t',precision_score(Y_test_1,y_pred_ts))

print('\n recall_score_train\t:\t',recall_score(Y_train_1,y_pred_tr))
print('\n recall_score_test\t:\t',recall_score(Y_test_1,y_pred_ts))

kstable_train=ks(pd.concat([pd.DataFrame(Y_train_1).reset_index(drop = True),pd.DataFrame(y_pred_tr_prob)], axis = 1),'Target',0)
kstable_test=ks(pd.concat([pd.DataFrame(Y_test_1).reset_index(drop = True),pd.DataFrame(y_pred_ts_prob)], axis = 1),'Target',0)

### XGBoost

In [ ]:
from numpy import loadtxt
from xgboost import XGBClassifier

from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import roc_auc_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import accuracy_score

model = XGBClassifier(objective='binary:logistic', booster='gbtree',learning_rate=0.1,max_depth=6,
                      n_estimators=110, n_jobs=6,num_parallel_tree=4, eval_metric='auc',tree_method='hist',
                      grow_policy='lossguide', subsample=0.5)
model.fit(X_train_1.values, Y_train_1)

# make predictions for train data
y_pred_XG_train = model.predict_proba(X_train_1.values)[:,1]
# make predictions for test data
y_pred_XG_test = model.predict_proba(X_test_1.values)[:,1]
# make predictions for validation data

y_pred_tr = model.predict(X_train_1.values)
y_pred_ts = model.predict(X_test_1.values)

print(pd.DataFrame.from_dict(zip(list(X_train_1.columns),model.feature_importances_*100)))

print('\nAUC Score for Train set\t:\t',roc_auc_score(Y_train_1,y_pred_XG_train))
print('\nAUC Score for Test set\t:\t',roc_auc_score(Y_test_1,y_pred_XG_test))

print('\n Accuracy_train\t:\t',accuracy_score(Y_train_1,y_pred_tr))
print('\n Accuracy_test\t:\t',accuracy_score(Y_test_1,y_pred_ts))

print('\n confusion_matrix_train\n',confusion_matrix(Y_train_1,y_pred_tr))
print('\n confusion_matrix_test\n',confusion_matrix(Y_test_1,y_pred_ts))

print('\n precision_score_train\t:\t',precision_score(Y_train_1,y_pred_tr))
print('\n precision_score_test\t:\t',precision_score(Y_test_1,y_pred_ts))

print('\n recall_score_train\t:\t',recall_score(Y_train_1,y_pred_tr))
print('\n recall_score_test\t:\t',recall_score(Y_test_1,y_pred_ts))

kstable_train_1=ks(pd.concat([pd.DataFrame(Y_train_1).reset_index(drop = True),pd.DataFrame(y_pred_XG_train)], axis = 1),'Target',0)
kstable_test_1=ks(pd.concat([pd.DataFrame(Y_test_1).reset_index(drop = True),pd.DataFrame(y_pred_XG_test)], axis = 1),'Target',0)

In [ ]:
# Define the model
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss') ## Define a different Model to other algorithms

# Create the parameter grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.1, 0.2],
    'max_depth': [3, 4, 5],
    'colsample_bytree': [0.3, 0.7, 1.0],
    'subsample': [0.6, 0.8, 1.0]
}

# Setup GridSearchCV
grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='accuracy', cv=3, verbose=1)

# Fit GridSearchCV
grid_search.fit(X_train, y_train)

# Best parameters and score
print("Best parameters found: ", grid_search.best_params_)
print("Best accuracy found: ", grid_search.best_score_)

In [ ]:
# Define the model
model = xgb.XGBClassifier(use_label_encoder=False, eval_metric='mlogloss') ## Define a different Model to other algorithms

# Define the parameter distributions
param_distributions = {
    'n_estimators': np.arange(50, 400, 50),
    'learning_rate': np.linspace(0.01, 0.6, 10),
    'max_depth': np.arange(3, 10),
    'colsample_bytree': np.linspace(0.3, 1.0, 8),
    'subsample': np.linspace(0.5, 1.0, 6)
}

# Setup RandomizedSearchCV
random_search = RandomizedSearchCV(estimator=model, param_distributions=param_distributions, n_iter=100, scoring='accuracy', cv=3, verbose=1, random_state=42)

# Fit RandomizedSearchCV
random_search.fit(X_train, y_train)

# Best parameters and score
print("Best parameters found: ", random_search.best_params_)
print("Best accuracy found: ", random_search.best_score_)

### Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_dtc = DecisionTreeClassifier(max_depth=3)
model_dtc.fit(X_train_1, Y_train_1)


# make predictions for train data
y_pred_dct_train = model_dtc.predict_proba(X_train_1)[:,1]
# make predictions for test data
y_pred_dct_test = model_dtc.predict_proba(X_test_1)[:,1]

y_pred_tr = model_dtc.predict(X_train_1)
y_pred_ts = model_dtc.predict(X_test_1)


print(pd.DataFrame.from_dict(zip(list(X_train_1.columns),model_dtc.feature_importances_*100)))

print('\nAUC Score for Train set\t:\t',roc_auc_score(Y_train_1,y_pred_dct_train))
print('\nAUC Score for Test set\t:\t',roc_auc_score(Y_test_1,y_pred_dct_test))

print('\n Accuracy_train\t:\t',accuracy_score(Y_train_1,y_pred_tr))
print('\n Accuracy_test\t:\t',accuracy_score(Y_test_1,y_pred_ts))

print('\n confusion_matrix_train\n',confusion_matrix(Y_train_1,y_pred_tr))
print('\n confusion_matrix_test\n',confusion_matrix(Y_test_1,y_pred_ts))

print('\n precision_score_train\t:\t',precision_score(Y_train_1,y_pred_tr))
print('\n precision_score_test\t:\t',precision_score(Y_test_1,y_pred_ts))

print('\n recall_score_train\t:\t',recall_score(Y_train_1,y_pred_tr))
print('\n recall_score_test\t:\t',recall_score(Y_test_1,y_pred_ts))

### Random Forest

In [ ]:
from sklearn.tree import DecisionTreeClassifier

model_rf = RandomForestClassifier(max_depth=7,n_estimators = 500, n_jobs = 4,max_features='sqrt',
                                  min_samples_split=6,min_samples_leaf=5, criterion = 'gini')
model_rf.fit(X_train_1, Y_train_1)

# make predictions for train data
y_pred_dct_train = model_rf.predict_proba(X_train_1)[:,1]
# make predictions for test data
y_pred_dct_test = model_rf.predict_proba(X_test_1)[:,1]

y_pred_tr = model_rf.predict(X_train_1)
y_pred_ts = model_rf.predict(X_test_1)

print(pd.DataFrame.from_dict(zip(list(X_train_1.columns),model_rf.feature_importances_*100)))

print('\nAUC Score for Train set\t:\t',roc_auc_score(Y_train_1,y_pred_dct_train))
print('\nAUC Score for Test set\t:\t',roc_auc_score(Y_test_1,y_pred_dct_test))

print('\n Accuracy_train\t:\t',accuracy_score(Y_train_1,y_pred_tr))
print('\n Accuracy_test\t:\t',accuracy_score(Y_test_1,y_pred_ts))

print('\n confusion_matrix_train\n',confusion_matrix(Y_train_1,y_pred_tr))
print('\n confusion_matrix_test\n',confusion_matrix(Y_test_1,y_pred_ts))

print('\n precision_score_train\t:\t',precision_score(Y_train_1,y_pred_tr))
print('\n precision_score_test\t:\t',precision_score(Y_test_1,y_pred_ts))

print('\n recall_score_train\t:\t',recall_score(Y_train_1,y_pred_tr))
print('\n recall_score_test\t:\t',recall_score(Y_test_1,y_pred_ts))

### All Baseline Models

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score
model={"log":LogisticRegression(),
       "dt":DecisionTreeClassifier(criterion='gini',
                                   max_depth = 9,
                                   min_samples_leaf = 5,
                                   class_weight = {1:2.65},
                                   min_samples_split = 4),
       "rf":RandomForestClassifier(max_depth=7,
                                   n_estimators = 500,
                                   n_jobs = 4,
                                   max_features='sqrt',
                                   min_samples_split=6,
                                   min_samples_leaf=5,
                                   criterion = 'gini'),
       "xgb":XGBClassifier(objective='binary:logistic',
                           booster='gbtree',
                           learning_rate=0.057,
                           max_depth=4,
                           min_child_weight=4,
                           n_estimators=500,
                           n_jobs=6,num_parallel_tree=6, eval_metric='auc',tree_method='hist',
                           grow_policy='lossguide', subsample=0.5, scale_pos_weight = 100),
       "nb":GaussianNB(var_smoothing=0.05),
       "knn":KNeighborsClassifier(n_neighbors=3)

}

for xgb in model:
    print('\n',model[xgb])
    model[xgb].fit(X_train1,y_train)
    y_train_pred=model[xgb].predict(X_train1)
    y_test_pred=model[xgb].predict(X_test1)

    print('roc_auc_score_train\t',roc_auc_score(y_train, y_train_pred))
    print('roc_auc_score_test\t',roc_auc_score(y_test, y_test_pred))
    print()
    print()

### AUC Plot

In [ ]:
from sklearn.metrics import roc_curve

false_positive_rate1, true_positive_rate1, threshold1 = roc_curve(df_train.Target, y_pred_XG_train)

plt.subplots(1, figsize=(10,10))
plt.title('Receiver Operating Characteristic - XG_boost')
plt.plot(false_positive_rate1, true_positive_rate1)
plt.plot([0, 1], ls="--")
plt.plot([0, 0], [1, 0] , c=".7"), plt.plot([1, 1] , c=".7")
plt.ylabel('True Positive Rate')
plt.xlabel('False Positive Rate')
plt.show()

### Speci vs sensi plot

In [ ]:
y_ot_pred_final = pd.DataFrame(df_train['Target']).reset_index(drop = True)
y_ot_pred_final['prob'] = y_pred_tr_prob
y_ot_pred_final

In [ ]:
numbers = [float(x)/10 for x in range(10)]
for i in numbers:
    y_ot_pred_final[i]= y_ot_pred_final.prob.map(lambda x: 1 if x > i else 0)
y_ot_pred_final.head()

In [ ]:
cutoff_df = pd.DataFrame( columns = ['prob','accuracy','sensi','speci','precision','recall'])
num = [0.0,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9]
for i in num:
    cm1 = confusion_matrix(y_ot_pred_final.NBO_Target, y_ot_pred_final[i] )
    total1=sum(sum(cm1))
    accuracy = (cm1[0,0]+cm1[1,1])/total1

    speci = cm1[0,0]/(cm1[0,0]+cm1[0,1])
    sensi = cm1[1,1]/(cm1[1,0]+cm1[1,1])
    precision=cm1[1,1]/(cm1[1,1]+cm1[0,1])
    recall=cm1[1,1]/(cm1[1,1]+cm1[1,0])
    cutoff_df.loc[i] =[ i ,accuracy,sensi,speci,precision,recall]
print(cutoff_df)

### Saving pickle Files

In [ ]:
import pickle
filename = 'Model_Name.sav'
pickle.dump(model, open(filename, 'wb'))

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
# Load data
iris = load_iris()
X, y = iris.data, iris.target

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Convert arrays to PyTorch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)  # use torch.long for classification labels
y_test = torch.tensor(y_test, dtype=torch.long)

# Create datasets
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

# Create data loaders
train_loader = DataLoader(dataset=train_dataset, batch_size=10, shuffle=True)
test_loader = DataLoader(dataset=test_dataset, batch_size=10, shuffle=False)

In [ ]:
class NeuralNet(nn.Module):
    def __init__(self):
        super(NeuralNet, self).__init__()
        self.fc1 = nn.Linear(4, 10)  # Input layer (4 features) to hidden layer (10 nodes)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(10, 3)  # Hidden layer to output layer (3 classes)

    def forward(self, x):
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# Initialize the network
model = NeuralNet()

In [ ]:
criterion = nn.CrossEntropyLoss()  # Suitable for classification tasks
optimizer = optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# Training loop
num_epochs = 100
for epoch in range(num_epochs):
    for inputs, labels in train_loader:
        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        
        # Backward and optimize
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
    if (epoch+1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

In [ ]:
# Set the model to evaluation mode
model.eval()
with torch.no_grad():
    correct = 0
    total = 0
    for inputs, labels in test_loader:
        outputs = model(inputs)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    print(f'Accuracy of the model on the test set: {100 * correct / total:.2f}%')

In [ ]:
# Predicting any outputs
with torch.no_grad():
    sample_data = torch.tensor([X_test[0]], dtype=torch.float32)  # Use the first test sample
    predicted = model(sample_data)
    print(f'Predicted value: {predicted.item()} Actual value: {y_test[0].item()}')